In [44]:
"""
Training Instruction:
    - Download the LungHist700 dataset and upload the file into the `digipath/` folder
    - Set the path for `csv_file`, `root_dir`, `load_file` in the `train()` from `digipath/src/train.py` with model configurations
"""

%rm -rf digipath
%rm -rf data

In [38]:
!git clone -b feature https://github.com/chuch3/digipath.git

fatal: destination path 'digipath' already exists and is not an empty directory.


In [43]:
%cd /content
%ls
!unzip LungHist700.zip

/content
digipath/  drive/  LungHist700.zip  sample_data/
Archive:  LungHist700.zip
   creating: data/
   creating: data/LungHist700/
  inflating: data/LungHist700/data.csv  
   creating: data/LungHist700/images/
   creating: data/LungHist700/images/aca_md/
  inflating: data/LungHist700/images/aca_md/aca_md_40x_95.jpg  
  inflating: data/LungHist700/images/aca_md/aca_md_20x_905.jpg  
  inflating: data/LungHist700/images/aca_md/aca_md_20x_73.jpg  
  inflating: data/LungHist700/images/aca_md/aca_md_40x_126.jpg  
  inflating: data/LungHist700/images/aca_md/aca_md_40x_303.jpg  
  inflating: data/LungHist700/images/aca_md/aca_md_20x_301.jpg  
  inflating: data/LungHist700/images/aca_md/aca_md_40x_402.jpg  
  inflating: data/LungHist700/images/aca_md/aca_md_20x_903.jpg  
  inflating: data/LungHist700/images/aca_md/aca_md_40x_85.jpg  
  inflating: data/LungHist700/images/aca_md/aca_md_20x_904.jpg  
  inflating: data/LungHist700/images/aca_md/aca_md_40x_306.jpg  
  inflating: data/LungHist700/

In [23]:
%mv data/ digipath/

In [41]:
%cd digipath/src/
os.getcwd()

/content/digipath/src


'/content/digipath/src'

In [42]:
!git pull

Already up to date.


In [2]:
import os
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import rich
import torch
import torch.nn as nn
import torchvision
from matplotlib.pyplot import imshow
from PIL import Image
from rich.panel import Panel
from sklearn.model_selection import GroupShuffleSplit
from torch.optim import Adam
from torch.optim.lr_scheduler import CosineAnnealingLR
from torch.utils.data import DataLoader, Dataset, TensorDataset
from torchvision import transforms
from torchvision.models import ViT_B_16_Weights
from tqdm import tqdm

from constant import (
    LUNG_IMAGES_DIR,
    LUNG_LOADED_FILE,
    LUNG_METADATA_FILE,
    LUNG_MODEL_DIR,
    LUNG_PREPROCESS_DIR,
    SSL_CHECKPOINT,
)
from dataset import load_dataset
from extract import extract_embeddings
from macenko import build_macenko_normalizer
from model import ClassifierHead
from ssl_train import pretrain_ssl
from transform import det_transform, rand_transform


In [6]:
csv_file = LUNG_METADATA_FILE
meta = pd.read_csv(csv_file)
meta

,superclass,subclass,resolution,image_id,patient_id
0,aca,bd,40x,901,1
1,aca,bd,40x,902,1
2,aca,bd,40x,903,1
3,aca,bd,40x,904,1
4,aca,bd,40x,905,1
...,...,...,...,...,...
686,nor,NaN,20x,901,45
687,nor,NaN,20x,902,45
688,nor,NaN,20x,903,45
689,nor,NaN,20x,904,45


In [9]:
meta.info()

<class 'pandas.DataFrame'>
RangeIndex: 691 entries, 0 to 690
Data columns (total 5 columns):
 #   Column      Non-Null Count  Dtype
---  ------      --------------  -----
 0   superclass  691 non-null    str  
 1   subclass    540 non-null    str  
 2   resolution  691 non-null    str  
 3   image_id    691 non-null    int64
 4   patient_id  691 non-null    int64
dtypes: int64(2), str(3)
memory usage: 27.1 KB


In [27]:
meta[['superclass', 'subclass', 'resolution']].value_counts()

superclass  subclass  resolution
aca         bd        20x           57
scc         bd        20x           50
                      40x           49
            pd        20x           48
                      40x           47
aca         bd        40x           46
            md        40x           46
            pd        20x           45
            md        20x           44
            pd        40x           42
scc         md        40x           36
                      20x           30
Name: count, dtype: int64

In [26]:
meta['subclass'].value_counts()

subclass
bd    202
pd    182
md    156
Name: count, dtype: int64

In [10]:
meta.isnull().sum()

superclass      0
subclass      151
resolution      0
image_id        0
patient_id      0
dtype: int64

In [15]:
meta[meta['superclass'] == 'nor'][['superclass', 'subclass']].head()

,superclass,subclass
540,nor,NaN
541,nor,NaN
542,nor,NaN
543,nor,NaN
544,nor,NaN


In [11]:
meta['resolution'].unique()

<StringArray>
['40x', '20x']
Length: 2, dtype: str

In [19]:
load_file=LUNG_LOADED_FILE
load_meta = pd.read_csv(load_file)
load_meta

,path,label,patient_id
0,/home/chu/dev/wsi-grading/data/LungHist700/ima...,1,16
1,/home/chu/dev/wsi-grading/data/LungHist700/ima...,1,2
2,/home/chu/dev/wsi-grading/data/LungHist700/ima...,1,16
3,/home/chu/dev/wsi-grading/data/LungHist700/ima...,1,15
4,/home/chu/dev/wsi-grading/data/LungHist700/ima...,1,12
...,...,...,...
686,/home/chu/dev/wsi-grading/data/LungHist700/ima...,0,7
687,/home/chu/dev/wsi-grading/data/LungHist700/ima...,0,10
688,/home/chu/dev/wsi-grading/data/LungHist700/ima...,0,6
689,/home/chu/dev/wsi-grading/data/LungHist700/ima...,0,12


In [21]:
load_meta['path'][0]

'/home/chu/dev/wsi-grading/data/LungHist700/images/aca_md/aca_md_40x_95.jpg'

In [1]:
from training import train
train()

=> Using device : cpu
=> Metadata file : `/home/chu/dev/wsi-grading/data/LungHist700/data.csv` 
=> Root directory : `/home/chu/dev/wsi-grading/data/LungHist700/images`


╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ LungHist700 Target Label Statistics                                                                             │
│ ----------                                                                                                      │
│ Label map: {'aca_bd': 0, 'aca_md': 1, 'aca_pd': 2, 'scc_bd': 3, 'scc_md': 4, 'scc_pd': 5, 'nor': 6}             │
│                                                                                                                 │
│ label                                                                                                           │
│ 6    151                                                                                                        │
│ 0    103                                                                                                        │
│ 3     99                                                                                                        │
│ 5     95                                                                                                        │
│ 1     90                                                                                                        │
│ 2     87                                                                                                        │
│ 4     66                                                                                                        │
│ Name: count, dtype: int64                                                                                       │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ LungHist700 Dataset Split Statistics (Patient-Level)                                                            │
│ ----------                                                                                                      │
│ Train split : 71.3459%                                                                                          │
│ Validation splt : 16.6425%                                                                                      │
│ Test splt : 12.0116%                                                                                            │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

=> Checking if loaded dataset exists.
=> Loaded dataset already exists, continuing process.
=> Loading cached embeddings from /home/chu/dev/wsi-grading/preprocess/lung_embedded_stain_ssl.pt


> Training~ :   1%|█▏                                                                                                                   | 2/200 [00:00<00:28,  6.90it/s]

...

 Epochs 195 | Train loss : 0.59 | Train acc 80.53 % | Val loss 0.5765 | Val acc 79.52%
 Epochs 196 | Train loss : 0.56 | Train acc 81.95 % | Val loss 0.5650 | Val acc 80.72%


> Training~ :  99%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████▊ | 198/200 [00:25<00:00,  7.85it/s]

 Epochs 197 | Train loss : 0.59 | Train acc 79.11 % | Val loss 0.5664 | Val acc 80.72%
 Epochs 198 | Train loss : 0.57 | Train acc 81.14 % | Val loss 0.5630 | Val acc 80.72%


> Training~ : 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 200/200 [00:26<00:00,  7.61it/s]

 Epochs 199 | Train loss : 0.57 | Train acc 81.34 % | Val loss 0.5580 | Val acc 80.72%
 Epochs 200 | Train loss : 0.57 | Train acc 81.54 % | Val loss 0.5510 | Val acc 80.72%

=> Final Test accuracy: 82.61% 

